# 09 · Exceptions and logging

| Session | Time | Previous | Next |
| --- | --- | --- | --- |
| Day 2 | about 45 minutes live | [08 · Design and architecture](08_design_and_architecture.ipynb) | [10 · Testing with pytest](10_testing_with_pytest.ipynb) |

Every month a new file of customers arrives and the churn model scores it. Sometimes the file is broken: a
column was renamed in the export, a value is negative, a field is empty. The warm-up notebook would either crash
with a cryptic `KeyError` deep inside pandas, or, worse, quietly produce wrong predictions.

This notebook is about making failures **loud, specific and traceable**: exceptions tell the caller *what*
went wrong, logging tells you later *what happened* during a run that nobody was watching.

> **Learning goals.** By the end you can:
>
> - explain why raising an exception is better than returning `None` or printing a message
> - choose a fitting built-in exception and design a small exception hierarchy for a project
> - decide where an exception should be caught, and keep the original cause with `raise ... from`
> - set up logging correctly: named loggers in modules, configuration once in the application

**Before you start:** notebooks 01 to 04 (classes and inheritance). Notebook 07 helps but is not required.

> **How to work on the exercises**
>
> - Every exercise is done at the computer, **alone or in pairs**. In pairs, one person types (the driver) and the other reviews and thinks ahead (the navigator); swap at every exercise. Online, the driver shares their screen, or you both use VS Code Live Share.
> - Levels: **Core** (everyone), **Stretch** (if you finish early), **Challenge** (for experienced programmers).
> - Each exercise ends with a **Discuss** question. Talk it through with your partner or the group. Working alone? Write a short answer in a markdown cell.
> - Check cells fail until the exercise is solved; that is expected. Hints and solutions are in the collapsed blocks.
> - **Check your understanding** questions: predict the output first, then run the cell below the question.

**In this notebook**

1. Fail loudly
2. Choosing a built-in exception
3. A small exception hierarchy for the project
4. Keep the cause: `raise ... from`
5. Where to catch
6. Logging

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path("../data/raw/telco_churn.csv")
raw = pd.read_csv(DATA_PATH)
raw.shape

---

## 1. Fail loudly

Since notebook 01 you have used `raise` and `try`/`except` in their simple form, mostly in exercise starters and check cells. This notebook is about using them well: which error to raise, how to design your own, and where to catch them.

Here is a helper from a typical notebook. It converts the `TotalCharges` column, which the export stores as
text, into numbers.

In [ ]:
def parse_total_charges_quiet(df: pd.DataFrame):
    try:
        return pd.to_numeric(df["TotalCharges"])
    except Exception:
        print("Could not parse TotalCharges")
        return None


result = parse_total_charges_quiet(raw)
print(type(result))

It printed a message and returned `None`. Three things are wrong with that:

1. **The caller cannot react.** The message went to the screen, not to the code that called the function. In a
   scheduled job nobody reads the screen.
2. **The failure moves somewhere else.** The next line, maybe `result.mean()`, crashes with
   `AttributeError: 'NoneType' object has no attribute 'mean'`, far away from the real cause.
3. **The real cause is lost.** Which rows? Which values? `except Exception` swallowed the original error.

An **exception** solves all three. `raise` stops the function immediately and travels up the call stack until
some code that knows what to do with it catches it. If nobody does, the program stops with a traceback that
points at the cause.

Let us look at the real reason first: the 11 blank values belong to customers with `tenure == 0`.

In [ ]:
blank = raw["TotalCharges"].str.strip() == ""
raw.loc[blank, ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]].head()

These customers joined this month and have not been billed yet, so the correct value is 0. A blank for a
customer with a long tenure, however, would be a real data problem. A good function encodes exactly that
knowledge and raises a clear error for anything else.

In [ ]:
def parse_total_charges(df: pd.DataFrame) -> pd.Series:
    total = pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce")
    new_customers = df["tenure"] == 0
    unexplained = total.isna() & ~new_customers
    if unexplained.any():
        examples = df.loc[unexplained, "customerID"].head(3).tolist()
        raise ValueError(
            f"TotalCharges missing for {int(unexplained.sum())} customers with tenure > 0, "
            f"e.g. {examples}"
        )
    return total.fillna(0.0)


parse_total_charges(raw).describe()

In [ ]:
broken = raw.copy()
broken.loc[5, "TotalCharges"] = " "  # customer with tenure 2: this blank is not explained

try:
    parse_total_charges(broken)
except ValueError as exc:
    print(f"ValueError: {exc}")

The message names the problem, the count and examples. Whoever reads it can go and fix the export.

## 2. Choosing a built-in exception

Python already has exceptions with clear meanings. Use them before inventing your own.

| Exception | Raise it when | Churn example |
|---|---|---|
| `ValueError` | the type is right but the value is not acceptable | `test_size=1.5`, a negative `MonthlyCharges` |
| `TypeError` | the argument has the wrong type | a list passed where a DataFrame is expected |
| `KeyError` | a key or column is missing | `df["Contract"]` on a file without that column |
| `FileNotFoundError` | a file does not exist | the monthly export was not delivered |
| `RuntimeError` | the object is in the wrong state | `predict()` called before `fit()` |
| `NotImplementedError` | a method still has to be written | starter code in the exercises |

Pick the most specific one. Callers catch exceptions by type, so the type is part of your function's interface.

> **Check your understanding**
>
> A function `split(df, test_size)` receives `test_size="0.2"` (a string). Which exception fits best, and why
> not `ValueError`? Then predict which exception Python itself raises in the cell below, where the string is
> compared with numbers.
>
> <details><summary><b>Show answer</b></summary>
>
> `TypeError`: the problem is the type (a string instead of a float). `ValueError` is for a value of the right type
> that is not acceptable, such as `test_size=1.5`. Python agrees: comparing a string with a number raises
> `TypeError: '<' not supported between instances of 'int' and 'str'`.
> </details>

In [ ]:
# Predict the output first, then run this cell
test_size = "0.2"
try:
    print(0 < test_size < 1)
except (TypeError, ValueError) as exc:  # a tuple catches either type
    print(f"{type(exc).__name__}: {exc}")

---

## 3. A small exception hierarchy for the project

Built-in exceptions do not say *where* an error comes from. In a project you usually want two things:

- catch "anything that went wrong inside our churn code" in one place, without also catching unrelated bugs;
- tell specific problems apart and carry extra information, such as the list of missing columns.

A **hierarchy** of exception classes gives you both. An exception class is an ordinary class that inherits from
`Exception` (notebook 02): the inheritance tree decides which `except` clauses match.

```mermaid
classDiagram
    Exception <|-- ChurnError
    ChurnError <|-- DataValidationError
    DataValidationError <|-- SchemaError
    DataValidationError <|-- DataQualityError
    ChurnError <|-- NotFittedError
    RuntimeError <|-- NotFittedError
```

In [ ]:
class ChurnError(Exception):
    """Base class for every error raised by our churn code."""


class DataValidationError(ChurnError):
    """Input data does not meet the expectations of the pipeline."""


class SchemaError(DataValidationError):
    """Required columns are missing."""

    def __init__(self, missing_columns: list[str]) -> None:
        self.missing_columns = missing_columns
        super().__init__(f"Missing required columns: {missing_columns}")


class DataQualityError(DataValidationError):
    """Values are present but not usable."""


class NotFittedError(ChurnError, RuntimeError):
    """A model was used before fit() was called."""

Notes on the design:

- The base class needs no body. A docstring is enough; its whole job is to be catchable.
- `SchemaError` stores `missing_columns` as an attribute. Code that catches it can use the list directly,
  instead of parsing the message text.
- `NotFittedError` also inherits from `RuntimeError`, so code that only knows built-in exceptions still
  catches it. The same trick is used by scikit-learn's own `NotFittedError`.

The finished version of these classes lives in `src/churn/errors.py`.

In [ ]:
REQUIRED_COLUMNS = ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract"]


def validate_schema(df: pd.DataFrame, required: list[str] = REQUIRED_COLUMNS) -> None:
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise SchemaError(missing)


renamed_export = raw.rename(columns={"Contract": "ContractType"})

try:
    validate_schema(renamed_export)
except SchemaError as exc:
    print(f"{type(exc).__name__}: {exc}")
    print("Columns to look for in the export:", exc.missing_columns)

In [ ]:
# Catching the family: one except clause handles every data problem
for df in [renamed_export, broken]:
    try:
        validate_schema(df)
        parse_total_charges(df)
    except DataValidationError as exc:
        print(f"data problem ({type(exc).__name__}): {exc}")
    except ValueError as exc:
        print(f"other value problem: {exc}")

The second file raised a plain `ValueError` from `parse_total_charges`, so it fell through to the second
clause. In the exercises you will make it a `DataQualityError`, so that both data problems belong to the same
family.

> **Check your understanding**
>
> Which of these `except` clauses catch a `SchemaError`?
> `except Exception`, `except ChurnError`, `except DataQualityError`, `except ValueError`
>
> Predict, then run the cell below. `issubclass(A, B)` asks "is class A a subclass of B?", and `except B` catches
> an A exactly when the answer is `True`.
>
> <details><summary><b>Show answer</b></summary>
>
> `except Exception` and `except ChurnError`. An `except` clause matches the class and all its subclasses.
> `SchemaError` is not a `DataQualityError` (they are siblings) and not a `ValueError` (it never inherits from it).
> </details>

In [ ]:
# Predict the output first, then run this cell
for clause in [Exception, ChurnError, DataQualityError, ValueError]:
    print(f"except {clause.__name__:17s} catches SchemaError: {issubclass(SchemaError, clause)}")

---

## 4. Keep the cause: `raise ... from`

Sometimes you catch a low-level exception and raise a more meaningful one. Use `raise NewError(...) from exc`
so that the traceback shows both: your explanation and the original error.

In [ ]:
def read_monthly_file(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path)
    except FileNotFoundError as exc:
        raise DataValidationError(
            f"Monthly export {path.name} not found. Has the export job run?"
        ) from exc


try:
    read_monthly_file(Path("../data/raw/customers_2026_11.csv"))
except DataValidationError as exc:
    print(f"{type(exc).__name__}: {exc}")
    print(f"caused by {type(exc.__cause__).__name__}: {exc.__cause__}")

Without `from exc`, Python still shows the original error, but with the confusing line "During handling of the
above exception, another exception occurred", which suggests a second bug. With `from exc` it says "The above
exception was the direct cause of the following exception", which is exactly what happened.

## 5. Where to catch

Most functions should **not** catch exceptions at all. They raise, and the error travels up. Catch an exception
only where you can do something useful with it:

- **at the boundary** of the program: the command line entry point, the scheduled job, the API endpoint. There
  you decide: log it, return an exit code, skip the file, alert someone;
- when you can **recover**: retry a network call, fall back to a default that is really correct;
- when you **translate** it into a better error (section 4).

Two patterns to avoid:

```python
try:
    df = clean(df)
except:            # bare except: also catches KeyboardInterrupt and your own typos
    pass           # the pipeline continues with broken data and nobody knows
```

```python
def step(df):
    try:
        ...
    except SchemaError as exc:
        print(exc)
        raise      # catching only to print and re-raise, at every level: noise, the error is reported many times
```

The full structure of `try` for reference:

```python
try:
    model = load_model(path)          # code that may fail
except FileNotFoundError:
    model = train_new_model()         # only for errors you can handle
else:
    print("loaded existing model")    # runs only if nothing was raised
finally:
    release_lock()                    # always runs, error or not
```

Here is a boundary: a monthly job that processes several files. One broken file should not stop the others, but
every failure must be recorded. Note that it catches `ChurnError` (our known problems) and lets everything
else, such as a bug in our own code, crash loudly.

In [ ]:
def process_month(df: pd.DataFrame) -> int:
    validate_schema(df)
    return len(df)


def run_monthly_job(batches: dict[str, pd.DataFrame]) -> dict[str, str]:
    status = {}
    for name, df in batches.items():
        try:
            n = process_month(df)
        except ChurnError as exc:
            status[name] = f"skipped: {exc}"
        else:
            status[name] = f"ok: {n} customers"
    return status


run_monthly_job({"2026-09": raw.head(500), "2026-10": renamed_export.head(500)})

The job keeps the error message, but a dictionary of status strings is not where you want to look at 3 a.m.
That is what logging is for.

## 6. Logging

`print()` writes to the screen and that is all. **Logging** adds what you need for a job that runs without you:

- a **timestamp** and the **name** of the module that wrote the message;
- a **level** (how important it is), so you can show details while developing and only problems in production;
- **destinations** (screen, file, a log service) chosen in one place, without touching the code that logs.

### Levels

| Level | Use it for | Churn example |
|---|---|---|
| `DEBUG` | details for diagnosing a problem | "Encoded Contract into 3 columns" |
| `INFO` | milestones of a normal run | "Loaded 7043 customers", "Holdout recall 0.78" |
| `WARNING` | something unusual, the run continues | "12 customers with an unseen PaymentMethod" |
| `ERROR` | a step failed | "Batch 2026-10 skipped: missing column Contract" |
| `CRITICAL` | the whole program cannot continue | "No model file, cannot score anything" |

### The two rules

1. **Every module gets its own named logger:** `logger = logging.getLogger(__name__)` at the top of the file.
   `__name__` is the module's import path, e.g. `churn.data`, so every message says where it came from.
2. **Only the application configures logging, once.** The application is whatever starts the run: a script, the
   command line entry point, or this notebook. Library code (your package) only *emits* log records. If a
   library called `logging.basicConfig()` at import time, it would hijack the output settings of every program
   that imports it.

In this repository, `src/churn/data.py` creates a logger and logs, and only `src/churn/__main__.py` calls
`logging.basicConfig`.

In this notebook, the notebook is the application, so we configure logging here. `force=True` replaces any
handlers configured earlier, so re-running the cell does not print every message twice.

In [ ]:
import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,  # write log lines into the notebook output
    force=True,
)

# In a module you would write logging.getLogger(__name__). In a notebook __name__ is "__main__",
# so we pick a descriptive name instead.
logger = logging.getLogger("churn.monthly_job")

logger.debug("Not shown: the level is INFO")
logger.info("Loaded %d customers", len(raw))
logger.warning("%d customers have tenure 0 and no TotalCharges yet", int(blank.sum()))

Notice `logger.info("Loaded %d customers", len(raw))` instead of an f-string. The logger fills in the values only
if the message is actually emitted. For a `DEBUG` message inside a loop that is switched off in production,
that saves work, and log tools can group messages by their template.

Now the monthly job with logging. `logger.exception(...)` inside an `except` block logs at `ERROR` level **and**
attaches the traceback.

In [ ]:
def run_monthly_job(batches: dict[str, pd.DataFrame]) -> list[str]:
    succeeded = []
    for name, df in batches.items():
        try:
            n = process_month(df)
        except ChurnError:
            logger.exception("Batch %s skipped", name)
        else:
            logger.info("Batch %s processed: %d customers", name, n)
            succeeded.append(name)
    logger.info("Monthly job finished: %d of %d batches ok", len(succeeded), len(batches))
    return succeeded


run_monthly_job({"2026-09": raw.head(500), "2026-10": renamed_export.head(500)})

### print, logging or warnings?

- `print()`: output that *is* the result, for a human, right now (a summary at the end of a notebook).
- `logging`: a record of what happened during a run, for whoever operates or debugs it later.
- `warnings.warn()`: a message to the *programmer who uses your code*, e.g. a deprecated argument. Python shows
  each warning once per location and tools such as pytest can turn warnings into errors.

## Exercises

The exercises use a typical monthly scoring batch: new customers without the `Churn` column.

In [ ]:
batch = raw.drop(columns=["Churn"]).sample(200, random_state=42).reset_index(drop=True)
batch.head(3)

### Exercise 9.1
> **Core** · for everyone

Write `validate_batch(df)` for a monthly scoring batch. It returns nothing if the batch is fine and raises:

- `SchemaError` if any of `REQUIRED_COLUMNS` is missing (reuse `validate_schema`);
- `DataQualityError` if `MonthlyCharges` contains negative values (message: how many, and example customer IDs);
- `DataQualityError` if `TotalCharges` is blank for a customer with `tenure > 0`.

Then write `score_batch(df)`, the boundary function: it calls `validate_batch`, logs `INFO` with the number of
customers if everything is fine and returns that number. If validation fails it logs the problem with
`logger.exception` and **re-raises**, so the caller still knows the batch failed.

**Discuss.** `score_batch` logs the error and then re-raises it. Section 5 called logging and re-raising at every level noise. Why is it fine here? And who should decide whether the monthly job continues with the other batches: `score_batch` or its caller?

In [ ]:
# Exercise 9.1: your code here


def validate_batch(df: pd.DataFrame) -> None:
    raise NotImplementedError("Your code here")


def score_batch(df: pd.DataFrame) -> int:
    raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise 9.1: run after solving it
validate_batch(batch)  # a good batch passes silently
assert score_batch(batch) == 200

try:
    validate_batch(batch.drop(columns=["tenure"]))
    raise AssertionError("missing column not detected")
except SchemaError as exc:
    assert exc.missing_columns == ["tenure"]

negative = batch.copy()
negative.loc[[3, 7], "MonthlyCharges"] = -10.0
try:
    validate_batch(negative)
    raise AssertionError("negative charges not detected")
except DataQualityError as exc:
    assert "2 " in str(exc) and negative.loc[3, "customerID"] in str(exc)

gap = batch.copy()
gap.loc[gap.index[gap["tenure"] > 0][0], "TotalCharges"] = " "
try:
    validate_batch(gap)
    raise AssertionError("blank TotalCharges not detected")
except DataQualityError:
    pass

try:
    score_batch(negative)
    raise AssertionError("score_batch must re-raise")
except DataQualityError:
    pass

print("All checks passed")

<details><summary><b>Show hint</b></summary>

Build boolean masks such as `df["MonthlyCharges"] < 0`, then use `mask.sum()` for the count and
`df.loc[mask, "customerID"].head(3).tolist()` for examples. For the blank check, reuse the mask logic from
`parse_total_charges` in section 1. In `score_batch`, use `try` / `except DataValidationError` / `raise` (a bare
`raise` inside an except block re-raises the current exception unchanged).
</details>

<details><summary><b>Show solution</b></summary>

```python
def validate_batch(df: pd.DataFrame) -> None:
    validate_schema(df)

    negative = df["MonthlyCharges"] < 0
    if negative.any():
        examples = df.loc[negative, "customerID"].head(3).tolist()
        raise DataQualityError(
            f"{int(negative.sum())} customers with negative MonthlyCharges, e.g. {examples}"
        )

    total = pd.to_numeric(df["TotalCharges"].astype(str).str.strip(), errors="coerce")
    unexplained = total.isna() & (df["tenure"] > 0)
    if unexplained.any():
        examples = df.loc[unexplained, "customerID"].head(3).tolist()
        raise DataQualityError(
            f"TotalCharges missing for {int(unexplained.sum())} customers with tenure > 0, "
            f"e.g. {examples}"
        )


def score_batch(df: pd.DataFrame) -> int:
    try:
        validate_batch(df)
    except DataValidationError:
        logger.exception("Batch rejected")
        raise
    logger.info("Batch accepted: %d customers", len(df))
    return len(df)
```
</details>

### Exercise 9.2
> **Stretch** · if you finish early

Some exports use a comma as decimal separator by mistake (`"1,889.50"` or `"1889,50"`). Write
`to_number(series, column)` that converts a text column with `pd.to_numeric(series, errors="raise")`. If pandas
fails, raise a `DataQualityError` that names the column, **chained** to the pandas error with `from`. Also log
at `INFO` level how many values were converted.

**Discuss.** Should `to_number` repair `"1889,50"` automatically instead of failing? What could go wrong if it guesses (think of `"1,889"`), and who should make that decision?

In [ ]:
# Exercise 9.2: your code here


def to_number(series: pd.Series, column: str) -> pd.Series:
    raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise 9.2: run after solving it
assert to_number(pd.Series(["1.5", "2"]), "TotalCharges").tolist() == [1.5, 2.0]
try:
    to_number(pd.Series(["1.5", "1889,50"]), "TotalCharges")
    raise AssertionError("bad value not detected")
except DataQualityError as exc:
    assert "TotalCharges" in str(exc)
    assert isinstance(exc.__cause__, ValueError), "use raise ... from exc"
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
def to_number(series: pd.Series, column: str) -> pd.Series:
    try:
        numbers = pd.to_numeric(series, errors="raise")
    except ValueError as exc:
        raise DataQualityError(f"Column {column!r} contains values that are not numbers") from exc
    logger.info("Converted %d values in %s", len(numbers), column)
    return numbers
```
</details>

### Exercise 9.3
> **Challenge** · for experienced programmers

`validate_batch` stops at the first problem. The person fixing the export then fixes one issue, reruns, gets the
next error, and so on. For data validation it is often better to **report all problems at once**.

Python 3.11 added `ExceptionGroup`: one exception that carries several. You raise it with
`raise ExceptionGroup("message", [exc1, exc2])` and handle it with `except*`, which can pick out each type:

```python
try:
    ...
except* SchemaError as group:
    ...   # group.exceptions holds all SchemaErrors
except* DataQualityError as group:
    ...
```

Write `validate_batch_all(df)` that runs all checks, collects every problem as an exception object, and raises an
`ExceptionGroup` if there was at least one. Then handle a batch with two problems using `except*` and print each.

**Discuss.** When is "stop at the first error" better, and when is "collect everything" better? (Hint: if a
required column is missing, can the other checks even run?)

In [ ]:
# Exercise 9.3: your code here


def validate_batch_all(df: pd.DataFrame) -> None:
    raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise 9.3: run after solving it
two_problems = batch.copy()
two_problems.loc[3, "MonthlyCharges"] = -5.0
first_long_term = two_problems.index[two_problems["tenure"] > 0][0]
two_problems.loc[first_long_term, "TotalCharges"] = " "

validate_batch_all(batch)  # no problems, no exception
try:
    validate_batch_all(two_problems)
    raise AssertionError("no ExceptionGroup raised")
except ExceptionGroup as group:
    assert len(group.exceptions) == 2
    assert all(isinstance(e, DataQualityError) for e in group.exceptions)
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
def validate_batch_all(df: pd.DataFrame) -> None:
    # Without the columns the value checks cannot run: stop early in that case
    validate_schema(df)

    problems: list[Exception] = []
    negative = df["MonthlyCharges"] < 0
    if negative.any():
        problems.append(DataQualityError(f"{int(negative.sum())} negative MonthlyCharges"))

    total = pd.to_numeric(df["TotalCharges"].astype(str).str.strip(), errors="coerce")
    unexplained = total.isna() & (df["tenure"] > 0)
    if unexplained.any():
        problems.append(DataQualityError(f"{int(unexplained.sum())} blank TotalCharges with tenure > 0"))

    if problems:
        raise ExceptionGroup(f"{len(problems)} problems in batch", problems)


problem_batch = batch.copy()
problem_batch.loc[3, "MonthlyCharges"] = -5.0
problem_batch.loc[problem_batch.index[problem_batch["tenure"] > 0][0], "TotalCharges"] = " "

try:
    validate_batch_all(problem_batch)
except* DataQualityError as group:
    for exc in group.exceptions:
        print("quality problem:", exc)
```

Collecting everything is better when the checks are independent and a human fixes the data (one round trip
instead of many). Stopping early is better when later checks depend on earlier ones (no column, no value check)
or when the first problem makes the rest meaningless. This solution combines both: schema first, then all value
checks together. Libraries such as pandera and Great Expectations work the same way.
</details>

## Summary

- Raise an exception instead of printing or returning `None`: the caller can react and the cause is not lost.
- Use the most specific built-in exception; add a small project hierarchy (`ChurnError` and children) so callers
  can catch a whole family and read attributes such as `missing_columns`.
- `raise NewError(...) from exc` keeps the original cause in the traceback.
- Catch at the boundary (job, CLI, API), where you can decide what to do. Never `except: pass`.
- `logger = logging.getLogger(__name__)` in every module; configure logging once, in the application.
- Use `%s` placeholders in log calls and `logger.exception` inside `except` blocks.

**Common pitfalls:** catching `Exception` "to be safe"; logging and re-raising at every level; `basicConfig` in
library code; f-strings in log calls inside hot loops; error messages without the offending values.

## Going further

- Structured (JSON) logs: in production, logs are often shipped to a search tool. A formatter that writes one
  JSON object per line (for example with the `python-json-logger` package) makes them queryable by field.
- loguru offers a simpler logging API with sensible defaults. It is installed in
  this repo's dev environment. Try `from loguru import logger; logger.info("hello")`.
- Logging to a file with rotation: `logging.handlers.RotatingFileHandler`.
- Data validation libraries that collect problems per column: pandera.

---

## Sources and further reading

**Sources.** This notebook is based on:

- [The Python Tutorial, chapter 8: Errors and Exceptions](https://docs.python.org/3/tutorial/errors.html) and the [built-in exception hierarchy](https://docs.python.org/3/library/exceptions.html#exception-hierarchy).
- [Logging HOWTO](https://docs.python.org/3/howto/logging.html), especially [Configuring logging for a library](https://docs.python.org/3/howto/logging.html#configuring-logging-for-a-library), and the [Logging Cookbook](https://docs.python.org/3/howto/logging-cookbook.html).
- [PEP 654, Exception Groups and except*](https://peps.python.org/pep-0654/).

**For self-study.** Where to go next if you want to learn more:

- [Real Python: Logging in Python](https://realpython.com/python-logging/) (article, beginner to intermediate): handlers, formatters and configuration files.
- [loguru](https://loguru.readthedocs.io/) (library docs): a simpler logging API, installed in this repo's dev environment.
- [pandera](https://pandera.readthedocs.io/) (library docs): declarative validation of DataFrames that reports all problems at once.

---

[← 08 · Design and architecture](08_design_and_architecture.ipynb) &nbsp;|&nbsp; [Course overview](../README.md) &nbsp;|&nbsp; [10 · Testing with pytest →](10_testing_with_pytest.ipynb)